# RoomBeacon — Post-Silver Processing

This notebook consumes only `data/silver/rental_listings.parquet`. It prepares analytical and model-ready variables without repeating Bronze cleaning, changing Silver, fitting models, or presenting market rankings.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'crawler').is_dir() and (p / 'analytics').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env', override=False)

import pandas as pd
import duckdb
from IPython.display import display, Markdown
from analytics.duckdb.connection import resolve_runtime_path
from roombeacon_crawler.config.get_env import env
from notebooks.utils.location_analysis import haversine_distance_km

pd.set_option('display.max_columns', 20)
SEED = 42

## 01 Load Canonical Silver

The only input is the persisted canonical Silver Parquet checkpoint: `data/silver/rental_listings.parquet`.

In [ ]:
SILVER_PATH = resolve_runtime_path(env.processing.silver_dir) / 'rental_listings.parquet'
assert SILVER_PATH.exists(), f'Canonical Silver not found: {SILVER_PATH}'
silver_sql_path = str(SILVER_PATH).replace("'", "''")
with duckdb.connect(':memory:') as silver_reader:
    silver_df = silver_reader.execute(
        f"SELECT * FROM read_parquet('{silver_sql_path}')"
    ).df()
assert len(silver_df) > 0
assert silver_df.rental_post_id.notna().all() and silver_df.rental_post_id.is_unique
display(pd.Series({'path': str(SILVER_PATH), 'rows': len(silver_df), 'columns': len(silver_df.columns)}, name='value').to_frame())

## 02 Analytical Eligibility

Eligibility masks use existing Silver status fields; they do not rewrite quality evidence.


In [ ]:
processed_df = silver_df.copy()
processed_df['analytical_price_eligible'] = processed_df.price_amount_clean.notna()
processed_df['analytical_area_eligible'] = processed_df.area_value_clean.notna()
processed_df['model_base_eligible'] = (
    processed_df.analytical_price_eligible
    & processed_df.analytical_area_eligible
    & processed_df.row_quality_status.ne('REQUIRES_REVIEW')
)
display(processed_df[['analytical_price_eligible', 'analytical_area_eligible', 'model_base_eligible']].sum().to_frame('rows'))


## 03 Derived Numeric Variables

Derived analytical values remain outside the canonical Silver contract.


In [ ]:
price = pd.to_numeric(processed_df.price_amount_clean, errors='coerce')
area = pd.to_numeric(processed_df.area_value_clean, errors='coerce')
processed_df['price_per_area'] = price / area.where(area.gt(0))
processed_df.loc[~np.isfinite(processed_df.price_per_area), 'price_per_area'] = np.nan
display(processed_df.price_per_area.describe().to_frame())


## 04 Trusted Spatial-derived Variables

Distance is optional and computed only for coordinates already classified as trusted in Silver.


In [ ]:
REFERENCE_LOCATION = None
processed_df['distance_km'] = np.nan
if REFERENCE_LOCATION is not None:
    required = {'latitude', 'longitude'}
    if not required <= set(REFERENCE_LOCATION):
        raise ValueError('REFERENCE_LOCATION requires latitude and longitude')
    trusted = processed_df.has_trusted_coordinate.fillna(False)
    processed_df.loc[trusted, 'distance_km'] = haversine_distance_km(
        processed_df.loc[trusted, 'map_latitude'],
        processed_df.loc[trusted, 'map_longitude'],
        REFERENCE_LOCATION['latitude'],
        REFERENCE_LOCATION['longitude'],
    )
display(pd.Series({'trusted_coordinate_rows': int(processed_df.has_trusted_coordinate.sum()),
                   'distance_rows': int(processed_df.distance_km.notna().sum())}, name='rows').to_frame())


## 05 Categorical Preparation

Create deterministic categorical representations without learning encoders or fitting a model.


In [ ]:
categorical_fields = ['source_code', 'ward_current', 'district_text_extracted', 'row_quality_status']
for field in categorical_fields:
    processed_df[f'{field}_category'] = processed_df[field].astype('string').fillna('UNKNOWN').astype('category')
display(processed_df[[f'{field}_category' for field in categorical_fields]].dtypes.to_frame('dtype'))


## 06 Model-ready Dataset Preparation

Prepare an eligible dataset and deterministic split labels. This notebook does not train, evaluate, or compare models.


In [ ]:
model_columns = [
    'rental_post_id', 'source_code', 'ward_current', 'district_text_extracted',
    'price_amount_clean', 'area_value_clean', 'price_per_area', 'distance_km',
]
model_ready_df = processed_df.loc[processed_df.model_base_eligible, model_columns].copy()
selector = model_ready_df.rental_post_id.astype('int64').mod(10)
model_ready_df['dataset_split'] = np.select(
    [selector.lt(7), selector.lt(9)], ['train', 'validation'], default='test'
)
assert model_ready_df.rental_post_id.is_unique
display(model_ready_df.dataset_split.value_counts(dropna=False).to_frame('rows'))


## 07 Processing Summary

Post-Silver transformations are complete. No Silver materialization or model fitting occurs here.


In [ ]:
summary = pd.Series({
    'silver_input_rows': len(silver_df),
    'processed_rows': len(processed_df),
    'model_ready_rows': len(model_ready_df),
    'derived_column_count': len(set(processed_df.columns) - set(silver_df.columns)),
}, name='value')
assert len(processed_df) == len(silver_df)
display(summary.to_frame())